In [2]:
import os 
import re
import pandas as pd
import torch
import transformers
from transformers import DataCollatorWithPadding, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sklearn.model_selection import train_test_split

In [3]:
def process_matches(matches, instruction_template, txt_files):
    for match in matches:
        article_text = match.group(1).strip()
        lines = article_text.split('\n')
        
        if lines:
            instruction = lines[0].strip()
            answer = '\n'.join(lines[1:]).strip()
            
            txt_files.append({
                "Instruction": f"{instruction_template}: {instruction}",
                "Answer": answer
            })
    return txt_files

In [4]:
def collect_txt_files(root_path, business_topic=False):
    txt_files = []
    for root, dirs, files in os.walk(root_path):
        for file in files:
            if file.endswith('.txt'):
                file_path = os.path.join(root, file) 
                with open(file_path, 'rb') as f:
                    binary_content = f.read()
                content = binary_content.decode('utf-8', errors='ignore')
                if business_topic == True:
                    matches = list(re.finditer(r'(ГЛАВА\s+\d+[\.\:].*?)(?=\n\s*ГЛАВА\s+\d+[\.\:]|\Z)', content, re.DOTALL | re.MULTILINE))
                    instruction_template = "Расскрой эту тему в бизнесе"
                    txt_files = process_matches(matches, instruction_template, txt_files)
                else:
                    matches = list(re.finditer(r'(Глава\s+\d+[\.\:].*?)(?=\n\s*Глава\s+\d+[\.\:]|\Z)', content, re.DOTALL | re.MULTILINE))
                    if matches: 
                        txt_files = process_matches(matches, "Расскажи про эту Главу", txt_files)
                    else:
                        matches = re.finditer(r'(Статья\s+\d+[\.\:].*?)(?=\n\s*Статья\s+\d+[\.\:]|\Z)', content, re.DOTALL | re.MULTILINE)
                        txt_files = process_matches(matches, "Расскажи про эту статью", txt_files)
    return txt_files

In [5]:
files = collect_txt_files(root_path="/kaggle/input/datasets/polinafronek1/business-dataset", business_topic=True)

In [6]:
def preparing_dataset(dataset, idx, tokenizer):
    item = dataset[idx]
    
    prompt = f"Инструкция: {item['Instruction']}\nОтвет: "
    answer = item['Answer']
    
    full_text = f"{prompt}{answer}{tokenizer.eos_token}"
    
    prompt_encoding = tokenizer(
        prompt,
        truncation=True,
        max_length=512,
        padding=False,
        return_tensors=None
    )
    
    full_encoding = tokenizer(
        full_text,
        truncation=True,
        max_length=512,
        padding='max_length',
        return_tensors='pt'
    )
    
    prompt_length = len(prompt_encoding['input_ids'])
    
    labels = full_encoding['input_ids'].clone()
    labels[0, :prompt_length] = -100
    
    return {
        'input_ids': full_encoding['input_ids'].squeeze(0),
        'attention_mask': full_encoding['attention_mask'].squeeze(0),
        'labels': labels.squeeze(0)
    }
    

In [7]:
from transformers import DataCollatorForSeq2Seq
def initialization(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,                 
        bnb_4bit_compute_dtype=torch.bfloat16, 
        bnb_4bit_quant_type="nf4",           
        bnb_4bit_use_double_quant=True,     
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        quantization_config=quantization_config,
        device_map='auto',
        low_cpu_mem_usage=True,
    )

    data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    max_length=512
    )
    
    return tokenizer, model, data_collator
    

In [8]:
!pip install -U bitsandbytes>=0.46.1

In [9]:
#tokenizer, model, data_collator = initialization("Qwen/Qwen3.5-4B")

In [10]:
#model.save_pretrained('/kaggle/working/.virtual_documents')

In [11]:
from transformers import DataCollatorForSeq2Seq

model= AutoModelForCausalLM.from_pretrained(
    "/kaggle/input/datasets/polinafronek1/qwen3-8b",  
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B")

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    max_length=512
)

`torch_dtype` is deprecated! Use `dtype` instead!
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [12]:
from datasets import Dataset 

processed_dataset = []
for idx in range(len(files)):
    processed_file = preparing_dataset(files, idx, tokenizer)
    processed_dataset.append(processed_file)

train_data, test_data = train_test_split(
    processed_dataset, 
    test_size=0.1, 
    random_state=42
)

train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

In [13]:
from peft import LoraConfig, get_peft_model

def adding_lora(model,rank_dimension=8,lora_alpha=10,lora_dropout=0.05):
    LORA_TARGET_MODULES = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]

    peft_config = LoraConfig(
    r=rank_dimension,  
    lora_alpha=lora_alpha, 
    lora_dropout=lora_dropout,  
    bias="none",  
    target_modules=LORA_TARGET_MODULES,  
    task_type="CAUSAL_LM", 
    )

    model = get_peft_model(model, peft_config)

    return model

In [14]:
model = adding_lora(model)

In [15]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

def train(output_dir, train_data, test_data, data_collator, train_epochs=3, micro_bath_size=1, learning_rate=2e-4):
    training_arguments = transformers.TrainingArguments(
        per_device_train_batch_size=micro_bath_size, 
        gradient_accumulation_steps=4, 
        num_train_epochs=train_epochs, 
        learning_rate=learning_rate,  
        bf16=True, 
        weight_decay=0.01,
        max_grad_norm=0.3,
        logging_steps=True, 
        optim="adamw_torch", 
        eval_strategy="steps",
        save_strategy="steps", 
        output_dir=output_dir, 
        load_best_model_at_end=True, 
        metric_for_best_model="eval_loss"
    )

    trainer = Trainer(
        model=model,
        train_dataset=train_data,
        eval_dataset=test_data,
        args=training_arguments,
        data_collator=data_collator, 
        callbacks=[EarlyStoppingCallback(early_stopping_patience=5)] 
    )
    
    train_result = trainer.train()
    return model, train_result

In [16]:
model, train_result = train(output_dir="/kaggle/working/", train_data=train_dataset, test_data=test_dataset, data_collator=data_collator)

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2299: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Step,Training Loss,Validation Loss
1,3.096793,2.822158
2,3.004937,2.798893
3,2.812774,2.727529
4,2.783918,2.644061
5,2.516086,2.584316
6,2.455496,2.549257
7,2.529797,2.529167
8,2.598816,2.511776
9,2.352249,2.498310
10,2.366498,2.487978


In [20]:
model.save_pretrained("/kaggle/working/merged_model")

In [ ]:
from peft import PeftModel
tokenizer, base_model, data_collator = initialization("Qwen/Qwen3.5-4B")
model_сomb = PeftModel.from_pretrained(base_model, "/kaggle/input/datasets/polinafronek1/law-adapter")
merged_model = model_сomb.merge_and_unload()

In [ ]:
def conversation(model, question):
    messages = [
        {"role": "system", "content": "Ты - бизнес-консультант"},
        {"role": "user", "content": question}
    ]
    
    text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=3000,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response

In [ ]:
response = conversation(model=base_model, question="Какие главные юридические тонкости нужно знать для старта бизнеса?")
print(response)

In [ ]:
response = conversation(model=merged_model, question="Какие главные юридические тонкости нужно знать для старта бизнеса?")
print(response)